In [ ]:
import torch
import numpy as np
from board import Board
from torch import nn
import torch.optim as optim

criterion = nn.MSELoss()







pygame 2.6.1 (SDL 2.28.4, Python 3.11.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import torch

# This logic checks what hardware is available
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 13, kernel_size = 3, stride = 1, out_channels = 64, padding = 1)
        self.conv2 = nn.Conv2d(in_channels = 64, kernel_size = 3, stride = 1, out_channels = 128, padding = 1)
        self.stack = nn.Sequential(self.conv1, nn.ReLU(), self.conv2, nn.ReLU(), nn.Flatten(), nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, x):
        logits = self.stack(x)
        return logits

model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("chess_model.pth", map_location=device))
model.eval()

print(model)

NeuralNetwork(
  (conv1): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (stack): Sequential(
    (0): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=8192, out_features=256, bias=True)
    (6): ReLU()
    (7): Linear(in_features=256, out_features=1, bias=True)
  )
)


C:\Users\callu\AppData\Local\Temp\ipykernel_9436\1435147538.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("chess_model.pth", map_loca

In [7]:
import random

def steps(gs):
    moves = gs.get_all_legal_moves()
    if (random.random() < 0.20):
        return random.choice(moves)
    
    if (not (moves)):
        return (0, 0)
    scores = []
    
    for start, target in moves:
        currentBoards = gs.createCurrentBoards(gs.getBoards())
        gs.make_move(start, target)
        matrix = gs.createMatrix()
        tens = torch.from_numpy(matrix)
        tens = tens.float()
        tens = torch.unsqueeze(tens, 0)
        with torch.no_grad():
            score = model(tens).to(device)
        m_points = gs.materialScore()
        total_score = score.item() + (m_points / 10.0)
        
        
        scores.append(total_score)
        gs.setBoards(currentBoards)
        gs.setBoardArray()
        gs.switchColor()

    if (gs.pieceColor == 1):
        bestMove = np.argmax(scores)
    else:
        bestMove = np.argmin(scores)
    return moves[bestMove]
    

In [6]:
board = Board(model=model)

In [7]:
torch.save(model.state_dict(), "chess_model.pth")

In [ ]:
game = Board(model = model)
game.humanColor = 1
game.run()


--- AI is thinking for color -1 ---
Evaluating move 5/20...
Evaluating move 10/20...
Evaluating move 15/20...
Evaluating move 20/20...
AI Move Decided

--- AI is thinking for color -1 ---
Evaluating move 5/21...
Evaluating move 10/21...
Evaluating move 15/21...
Evaluating move 20/21...
AI Move Decided

--- AI is thinking for color -1 ---
Evaluating move 5/25...
Evaluating move 10/25...
Evaluating move 15/25...
Evaluating move 20/25...
Evaluating move 25/25...
AI Move Decided

--- AI is thinking for color -1 ---
Evaluating move 5/30...
Evaluating move 10/30...
Evaluating move 15/30...
Evaluating move 20/30...
Evaluating move 25/30...
Evaluating move 30/30...
AI Move Decided

--- AI is thinking for color -1 ---
Evaluating move 5/29...
Evaluating move 10/29...
Evaluating move 15/29...
Evaluating move 20/29...
Evaluating move 25/29...
AI Move Decided

--- AI is thinking for color -1 ---
Evaluating move 5/30...
Evaluating move 10/30...
Evaluating move 15/30...
Evaluating move 20/30...
Eval

In [ ]:


optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
episode = 0

try:
    while True:
        for episode in range(100):
            if (episode % 50 == 0):
                torch.save(model.state_dict(), "chess_model.pth")
                print(f"Checkpoint saved at episode {episode}")
            board = Board(model=model)
            in_progress = True
            moveCount = 0
            episode_history = []
            while in_progress:
                if (
                    board.check_game_status() == "CHECKMATE"
                    or board.check_game_status() == "DRAW"
                    or moveCount > 200
                ):
                    in_progress = False
                    break
                # print("Move Count: " + str(moveCount) + ", Start: " + str(bestMove[0]) + ", End: " + str(bestMove[1]))
                bestMove = steps(board)
                mat = board.createMatrix()
                episode_history.append(torch.from_numpy(mat))
                board.make_move(bestMove[0], bestMove[1])
                
                moveCount += 1
            if board.check_game_status() == "CHECKMATE":
                if board.pieceColor == 1:
                    outcome = -1.0
                else:
                    outcome = 1.0
            else:
                outcome = 0.0
            batchMatrix = torch.stack(episode_history).float()
            outcomeMatrix = torch.full((moveCount, 1), outcome).to(device)
            optimizer.zero_grad()
            predictions = model(batchMatrix)
            loss = criterion(predictions, outcomeMatrix)
            loss.backward()
            optimizer.step()
            #print("Loss: " + str(loss.item()) + ", Move Count: " + str(moveCount))
except KeyboardInterrupt:
    print("\nTraining interrupted by user. Saving progress...")
    torch.save(model.state_dict(), "chess_model_final.pth")
    print("Final weights saved. Safe to close kernel.")

Checkpoint saved at episode 0


IndexError: Cannot choose from an empty sequence

: 

In [5]:
# 1. Make sure model is initialized
model = NeuralNetwork().to(device)

# 2. Load the file
model.load_state_dict(torch.load("chess_model.pth", map_location=device))

# 3. Ensure you are in training mode
model.train()

C:\Users\callu\AppData\Local\Temp\ipykernel_9436\2219064621.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("chess_model.pth", map_locat

NeuralNetwork(
  (conv1): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (stack): Sequential(
    (0): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=8192, out_features=256, bias=True)
    (6): ReLU()
    (7): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [1]:
torch.save(model.state_dict(), "chess_model.pth")
print("Saved Latest Weights to chess_model.pth")

NameError: name 'torch' is not defined